# 第7章 概率、分布与抽样

第6章只记录了一条已经发生的财富路径。为了讨论未来的不确定性，项目需要区分透明假设、随机抽样和事实预测。

先写下亏20%、赚5%和赚30%三种结果，再反复抽样，观察平均数、分位数和尾部怎样变化。

![可能结果、概率模型、抽样与统计量关系](assets/course/07_probability_sampling.png)

这张图只说明模型、抽样和统计量之间的关系。最后，你要交出一份随机实验报告。下一章会用它检查风险。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"]=(8,4.5); plt.rcParams["axes.grid"]=True
plt.rcParams["font.sans-serif"]=["Arial Unicode MS","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"]=False
rng=np.random.default_rng(20260711)

## 7.1 随机变量：给结果赋数值

已有线索：回忆第1章“情景不是预测”和第6章一期收益率的含义。本节要解决：**结果、事件、随机变量、概率、总体和样本分别是什么？**

先按顺序认识六个对象，后面的公式才有明确含义：

1. **结果**：一次未来一年实验最后出现的具体状态，例如收益为$-20\%$、$5\%$或$30\%$。
2. **事件**：我们关心的一组结果，例如“发生亏损”在本例中包含$-20\%$这一结果。
3. **随机变量**：把每个结果对应成数值的规则。本章把一年收益记作$R$。
4. **概率**：模型赋给每个结果的权重，必须非负且总和为1。
5. **总体**：这张概率表所描述的全部可能性及其权重，不是一次实际观察表。
6. **样本**：按照模型实际抽到的一组有限结果；不同样本会有不同均值。

设$R$的三个结果为$-20\%$、$5\%$、$30\%$，概率分别为0.2、0.5、0.3。先逐行计算“结果$\times$概率”，再把三项相加得到期望：

$$E[R]=\sum_i p_i r_i.$$

方差再计算各结果与期望之差的加权平方；标准差是方差的平方根：

$$Var(R)=\sum_i p_i(r_i-E[R])^2.$$

动手前先做：判断7.5%的期望收益是否必须是三个情景之一，并先手算概率加权和。核对提示：逐行算`结果×概率`再求和；代码中先显示情景表，最后才使用向量乘法。

但是，期望是模型中心而非个人保证；总体是概率模型描述的全部可能性，不等于一张巨大数据表。


In [ ]:
outcomes = np.array([-.20, .05, .30])
probabilities = np.array([.2, .5, .3])

scenario_table = pd.DataFrame({
    "结果（收益）": outcomes,
    "概率（模型权重）": probabilities,
    "结果×概率": outcomes * probabilities,
})
display(scenario_table.style.format({
    "结果（收益）": "{:.1%}",
    "概率（模型权重）": "{:.1%}",
    "结果×概率": "{:.2%}",
}))

expected = np.sum(outcomes * probabilities)
variance = np.sum(probabilities * (outcomes - expected) ** 2)
print({
    "概率和": float(probabilities.sum()),
    "期望收益": f"{expected:.2%}",
    "方差（收益率平方）": round(float(variance), 6),
    "标准差": f"{np.sqrt(variance):.2%}",
})




### 我的解释

期望收益是否一定是三个情景之一？标准差为什么与收益率使用相同单位，而方差不是？

<!-- 在这里填写；完成前AI不要代答 -->

## 7.2 模拟不是“制造事实”

已有线索：指出三情景模型中哪些是结果数组，哪些是概率数组。本节要解决：**从指定概率模型抽样能回答什么，又不能证明什么？**

随机模拟从我们指定的概率模型中抽样。它能回答“如果模型成立会怎样”，不能证明模型符合市场。

动手前先做：预测抽10次是否必然恰好出现2、5、3次三种结果，再比较10、100、10,000次频率。核对提示：拆解`rng.choice(outcomes, size, p)`三个输入；先打印前12次抽样，再汇总次数和比例。

但是，随机种子用于重复得到计算，不会令假设更真实；模拟不能替代数据和机制检验。


In [ ]:
draws=rng.choice(outcomes,size=10_000,p=probabilities)
print({"模拟均值":draws.mean(),"理论期望":expected,"模拟亏损比例":np.mean(draws<0)})
plt.hist(draws,bins=[-.25,-.075,.175,.35],rwidth=.8); plt.xlabel("收益情景"); plt.ylabel("次数"); plt.title("离散情景的10000次抽样"); plt.show()

### 怎样读这张图：次数不是概率本身

1. 比较三根柱子的样本次数，而不是只看哪根最高。
2. 把次数除以10,000后再与`[0.2, 0.5, 0.3]`比较。
3. 更换随机种子会改变柱高；模型权重未改变。


## 7.3 大数定律：样本平均逐渐稳定

已有线索：回忆第7.2节每一次抽样仍可能亏损，以及第6章算术平均的计算。本节要解决：**样本量增加时，累计样本均值会怎样变化？**

大数定律不保证短期接近期望，也不说明期望本身安全；它说明在适当条件下，独立同分布样本平均随样本量增加趋近总体期望。

动手前先做：判断累计均值是否单调靠近期望，并比较三个随机种子的早期路径。核对提示：用`cumsum()/arange()`计算累计均值，并打印样本量10、100、1,000、5,000的检查点。

但是，大数定律不消除单次尾部损失；真实市场也可能不满足固定权重与相互独立。


In [ ]:
draws=rng.choice(outcomes,size=5000,p=probabilities)
running_mean=np.cumsum(draws)/np.arange(1,len(draws)+1)
plt.plot(running_mean,label="累计样本均值"); plt.axhline(expected,color="red",ls="--",label="理论期望")
plt.xscale("log"); plt.xlabel("样本量（对数轴）"); plt.ylabel("平均收益"); plt.title("大数定律的数值观察"); plt.legend(); plt.show()

### 怎样读这张图：稳定不等于单调

1. 先看前100次，找出累计均值多次穿过理论期望的位置。
2. 再看1,000次以后摆动范围是否缩小。
3. 横轴如果使用对数刻度，后期大量样本会被压缩；不要把视觉平滑误认为没有误差。


### 停下来核对

为什么前几十次波动剧烈？把横轴改为线性后，视觉感受有何变化？金融市场收益为何可能不满足独立同分布？

### 我的回答

<!-- 在这里填写；完成前AI不要代答 -->


## 7.4 分位数与尾部

已有线索：先把一组收益从小到大排序，并回忆布尔条件如何筛出小于阈值的元素。本节要解决：**不用假设正态分布，怎样描述最差的一小部分结果？**

5%分位数表示约5%的观察不高于该数，并不等于“最大损失”。经验分位数依赖样本和方法，在小样本下尤其不稳定。

动手前先做：预测均值和标准差接近的两个模型，其1%分位和极端事件比例是否也必须接近。核对提示：用`np.quantile`和布尔筛选；读对数纵轴时比较数量级，不比较柱形的表面高度。

但是，分位数是尾部入口而非最大损失；小样本下极端分位尤其不稳定。


In [ ]:
normal=rng.normal(.0003,.01,100_000)
heavy=rng.standard_t(df=4,size=100_000)*.01/np.sqrt(4/(4-2))+.0003
rows=[]
for name,x in [("正态",normal),("t(4)厚尾",heavy)]:
    rows.append({"模型":name,"均值":x.mean(),"标准差":x.std(ddof=1),"1%分位":np.quantile(x,.01),"绝对收益>4%":np.mean(np.abs(x)>.04)})
display(pd.DataFrame(rows).set_index("模型").style.format("{:.3%}"))

In [ ]:
fig,ax=plt.subplots(); bins=np.linspace(-.06,.06,120)
ax.hist(normal,bins=bins,density=True,alpha=.5,label="正态"); ax.hist(heavy,bins=bins,density=True,alpha=.5,label="t(4)厚尾")
ax.set_yscale("log"); ax.set(title="相同均值和方差附近、不同尾部（纵轴对数）",xlabel="收益",ylabel="密度"); ax.legend(); plt.show()

### 怎样读这张图：同中心、同尺度，不同尾部

1. 先核对上方表格中的均值和标准差是否接近。
2. 再沿对数纵轴比较两侧远离中心的位置，观察厚尾模型是否留下更多质量。
3. 图来自两个教学模型；不能仅凭形状宣布真实市场服从其中之一。


**量化编程警告**：样本均值和标准差相近，不代表极端风险相近。正态模型不是默认真理；模型选择应结合机制、诊断和压力测试。


## 7.5 抽样分布与中心极限定理

已有线索：按顺序复述：概率模型、一个大小为n的样本、样本均值、重复得到的许多均值。本节要解决：**为什么单日收益分布和样本均值的抽样分布不是同一个对象？**

重复抽取大小为$n$的样本，每次计算均值；这些均值本身构成抽样分布。许多条件下，样本均值标准误约为`总体标准差/sqrt(n)`。

动手前先做：预测`n=100`时均值分布的中心是否移动，以及宽度约为`n=1`时的多少。核对提示：二维数组形状是`(重复次数, 样本量)`；`axis=1`表示每一行计算一个样本均值。

但是，样本均值接近正态不表示单日收益变成正态；依赖、状态变化和极厚尾会削弱近似。


In [ ]:
population = rng.standard_t(df=4, size=500_000)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharex=True)
sampling_rows = []
for ax, n in zip(axes, [1, 10, 100]):
    means = rng.choice(population, size=(5000, n), replace=True).mean(axis=1)
    sampling_rows.append({"样本量": n, "样本均值的平均": means.mean(), "样本均值的标准差": means.std(ddof=1)})
    ax.hist(means, bins=60, density=True)
    ax.axvline(0, color="black", ls="--", lw=1)
    ax.set_title(f"样本量 n={n}")
    ax.set_xlabel("样本均值")
    ax.set_xlim(-4, 4)
axes[0].set_ylabel("密度")
fig.suptitle("样本均值的抽样分布（统一横轴）")
plt.tight_layout()
plt.show()
display(pd.DataFrame(sampling_rows).style.format({"样本均值的平均": "{:.3f}", "样本均值的标准差": "{:.3f}"}))


### 怎样读这张图：变窄的是样本均值分布

1. 三幅图使用同一横轴，比较中心位置是否明显移动。
2. 比较`n=1、10、100`的横向宽度，并核对打印出的标准差。
3. 这里变得集中的是“重复计算的样本均值”，不是单日收益本身。




### 我的解释

随着$n$增加，样本均值分布的中心和宽度如何变化？这是否说明单日收益本身变成正态？

<!-- 在这里填写；完成前AI不要代答 -->

## 7.6 编程练习：离散分布统计

已有线索：闭卷写出概率非负、概率和为1、结果与概率长度一致三条规则。本节要解决：**怎样让函数既计算统计量，又拒绝非法概率模型？**

检查概率合法性，返回期望、方差和标准差。

动手前先做：判断负概率、概率和不为1、含`NaN`和长度不一致各应触发什么结果。核对提示：A级验证输入，B级计算期望，C级返回有字段名的结果并添加确定性与对称分布测试。

但是，函数正确不等于概率设定符合现实；模型来源和估计误差仍需单独说明。


In [ ]:
def discrete_stats(outcomes,probabilities):
    # TODO
    return None

In [ ]:
ans=discrete_stats([-1,1],[.5,.5])
if ans is None: print("练习尚未完成。")
else: print("基础测试：",np.allclose(ans,[0,1,1]))

## 项目交付：随机实验报告

已有线索：用一句话分别定义总体、样本、统计量、抽样分布和模拟输出。本节要解决：**能否提交一份把假设、一次抽样、重复实验与结论边界分开的随机实验报告？**

自行设计一个三情景投资模型，计算理论统计量；模拟不同样本量；比较正态与厚尾模型的1%分位和极端事件比例；解释模拟结论依赖哪些假设。

**关键区分**：总体、样本、估计量和模拟输出不是同一个对象。

动手前先做：预测薄尾与厚尾模型在1%分位和极端事件比例上的差异，再运行实验。核对提示：固定种子并保存全部参数；表格同时报告理论值、模拟值、样本量和抽样误差。

分布描述可能性，但是还没有决定投资者最关心哪一种损失；这由第8章继续处理。
